# Advanced Grape Leaf Disease Analysis: Step-by-Step Pipeline

This notebook provides a robust, research-grade pipeline for grape leaf disease diagnosis. It is designed specifically for **Google Colab**.

### Pipeline Overview:
1. **Environment Setup**: Mount Drive and install dependencies.
2. **Segmentation**: Automated background removal to isolate leaves.
3. **Training**: Ensemble of DenseNet and EfficientNet backbones.
4. **Evaluation**: Statistical metrics, ROC curves, and Confusion Matrices.
5. **Severity Analysis**: Global infection calculation and CSV logging.
6. **Inference**: Visual diagnosis with highlighted borders.

## 1. Setup Environment

In [ ]:
# Install necessary libraries
# !pip install torch torchvision torchaudio scikit-learn matplotlib opencv-python pillow seaborn pandas tqdm

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import pandas as pd
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from PIL import Image
import copy
import shutil
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully!")
except:
    print("Manual mounting required or not in Colab.")

## 2. Configuration

In [ ]:
# --- UPDATE THESE PATHS ---
RAW_DATA_PATH = '/content/drive/MyDrive/GrapeDataset/Raw' 
BASE_OUTPUT_PATH = '/content/drive/MyDrive/GrapeDataset/Output'

SEGMENTED_DATA_PATH = os.path.join(BASE_OUTPUT_PATH, 'Segmented')
MODEL_SAVE_PATH = os.path.join(BASE_OUTPUT_PATH, 'Models')
RESULTS_PATH = os.path.join(BASE_OUTPUT_PATH, 'Results')

for path in [SEGMENTED_DATA_PATH, MODEL_SAVE_PATH, RESULTS_PATH]:
    os.makedirs(path, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 3. Dataset Pre-processing (Segmentation)

In [ ]:
def isolate_leaf(image_path):
    image = cv2.imread(image_path)
    if image is None: return None, None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([20, 30, 30]), np.array([100, 255, 255]))
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    leaf_only = cv2.bitwise_and(image_rgb, image_rgb, mask=mask)
    return leaf_only, mask

def process_dataset(src, dest):
    if not os.path.exists(src):
        print(f"Error: Source path {src} does not exist.")
        return
    
    categories = [d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d))]
    for cat in categories:
        print(f"Processing Category: {cat}")
        src_dir, dest_dir = os.path.join(src, cat), os.path.join(dest, cat)
        os.makedirs(dest_dir, exist_ok=True)
        
        imgs = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for img_name in tqdm(imgs):
            out_path = os.path.join(dest_dir, img_name)
            if os.path.exists(out_path): continue
            
            leaf, _ = isolate_leaf(os.path.join(src_dir, img_name))
            if leaf is not None:
                Image.fromarray(leaf).save(out_path)

# Execute Segmentation
process_dataset(RAW_DATA_PATH, SEGMENTED_DATA_PATH)

## 4. Model Definition and Persistence

In [ ]:
def get_baselines(num_classes):
    m1 = models.densenet121(weights='DEFAULT')
    m1.classifier = nn.Linear(m1.classifier.in_features, num_classes)
    
    m2 = models.efficientnet_b0(weights='DEFAULT')
    m2.classifier[1] = nn.Linear(m2.classifier[1].in_features, num_classes)
    return m1, m2

class SimpleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SimpleUNet, self).__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(), nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2 = conv_block(in_channels, 64), conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        u1 = self.up1(e2); d1 = self.dec1(torch.cat([u1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

def get_deeplabv3_plus(num_classes=1):
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, num_classes, 1)
    return model

def get_fcn_8s(num_classes=1):
    model = models.segmentation.fcn_resnet50(weights='DEFAULT')
    model.classifier[4] = nn.Conv2d(256, num_classes, 1)
    return model

class SoftEnsemble(nn.Module):
    def __init__(self, m1, m2): 
        super().__init__()
        self.m1, self.m2 = m1, m2
    def forward(self, x): 
        return (self.m1(x) + self.m2(x)) / 2

def save_weights(model, name):
    torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, f"{name}.pth"))

def load_weights(model, name):
    path = os.path.join(MODEL_SAVE_PATH, f"{name}.pth")
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        model.to(DEVICE)
        return True
    return False

## 5. Training Module

In [ ]:
def train_model(model, loaders, name, epochs=10):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    best_acc, best_wts = 0.0, None

    for epoch in range(epochs):
        print(f'Epoch {epoch}/{epochs-1}')
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            l_run, c_run = 0.0, 0
            
            for inputs, labels in loaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward(); optimizer.step()
                l_run += loss.item() * inputs.size(0)
                c_run += torch.sum(preds == labels.data)
            
            acc = c_run.double() / len(loaders[phase].dataset)
            history[f'{phase}_acc'].append(acc.item())
            history[f'{phase}_loss'].append(l_run / len(loaders[phase].dataset))
            
            if phase == 'val':
                print(f'Val Acc: {acc:.4f}')
                if acc > best_acc:
                    best_acc = acc
                    best_wts = copy.deepcopy(model.state_dict())
                    
    model.load_state_dict(best_wts)
    save_weights(model, name)
    return model, history

## 6. Execution: Load Data and Train

In [ ]:
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), 
                         transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
full_ds = datasets.ImageFolder(SEGMENTED_DATA_PATH, transform=tf)
train_idx, temp_idx = train_test_split(np.arange(len(full_ds)), test_size=0.3, stratify=full_ds.targets, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[full_ds.targets[i] for i in temp_idx], random_state=42)

loaders = {'train': DataLoader(Subset(full_ds, train_idx), batch_size=32, shuffle=True),
           'val': DataLoader(Subset(full_ds, val_idx), batch_size=32),
           'test': DataLoader(Subset(full_ds, test_idx), batch_size=32)}

densenet, efficientnet = get_baselines(len(full_ds.classes))

# 1. DenseNet
if not load_weights(densenet, 'densenet'):
    densenet, h1 = train_model(densenet, loaders, 'densenet')

# 2. EfficientNet
if not load_weights(efficientnet, 'efficientnet'):
    efficientnet, h2 = train_model(efficientnet, loaders, 'efficientnet')

ensemble = SoftEnsemble(densenet, efficientnet).to(DEVICE)

## 7. Comparative Evaluation and Plots

In [ ]:
def evaluate_all(models_dict, loader, classes):
    plt.figure(figsize=(10, 8))
    for name, model in models_dict.items():
        model.eval()
        y_true, y_pred, y_probs = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                out = model(inputs.to(DEVICE))
                y_true.extend(labels.numpy())
                y_pred.extend(torch.max(out, 1)[1].cpu().numpy())
                y_probs.extend(F.softmax(out, dim=1).cpu().numpy())
        
        print(f"\n--- {name} Classification Report ---")
        print(classification_report(y_true, y_pred, target_names=classes))
        
        # Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix: {name}'); plt.show()

        fpr, tpr, _ = roc_curve(np.eye(len(classes))[y_true].ravel(), np.array(y_probs).ravel())
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc(fpr, tpr):.2f})')
    
    plt.figure(figsize=(10, 8))
    plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.title('ROC Comparison'); plt.savefig(os.path.join(RESULTS_PATH, 'roc_curve.png')); plt.show()

evaluate_all({'Ensemble': ensemble, 'DenseNet': densenet, 'EfficientNet': efficientnet}, loaders['test'], full_ds.classes)

## 8. Severity Logging (CSV)

In [ ]:
def get_severity(leaf_rgb, mask):
    lab = cv2.cvtColor(leaf_rgb, cv2.COLOR_RGB2Lab)
    _, a, _ = cv2.split(lab)
    a_blurred = cv2.GaussianBlur(a, (5, 5), 0)
    _, d_mask = cv2.threshold(a_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    d_mask = cv2.bitwise_and(d_mask, d_mask, mask=mask)
    leaf_area = np.sum(mask > 0)
    sev = (np.sum(d_mask > 0) / leaf_area) * 100 if leaf_area > 0 else 0
    return d_mask, sev

print("Calculating severity for dataset...")
log = []
for path, label_idx in tqdm(full_ds.samples):
    leaf, mask = isolate_leaf(path)
    if leaf is not None:
        _, s = get_severity(leaf, mask)
        log.append({'image': os.path.basename(path), 'label': full_ds.classes[label_idx], 'severity_%': round(s, 2)})

pd.DataFrame(log).to_csv(os.path.join(RESULTS_PATH, 'severity_analysis.csv'), index=False)
print(f"CSV results saved to {RESULTS_PATH}")

## 9. Final Visual Diagnosis Example

In [ ]:
def visualize_final(image_path, model, classes):
    orig = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    leaf, mask = isolate_leaf(image_path)
    d_mask, sev = get_severity(leaf, mask)
    
    contours, _ = cv2.findContours(d_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    border_img = leaf.copy()
    cv2.drawContours(border_img, contours, -1, (255, 0, 0), 2)
    
    transform = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor(), 
                                    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    model.eval()
    with torch.no_grad():
        out = model(transform(leaf).unsqueeze(0).to(DEVICE))
        pred = classes[torch.max(out, 1)[1][0]]
    
    fig, ax = plt.subplots(1, 4, figsize=(24, 6))
    ax[0].imshow(orig); ax[0].set_title("Original")
    ax[1].imshow(leaf); ax[1].set_title("Isolated Leaf")
    ax[2].imshow(d_mask, cmap='hot'); ax[2].set_title("Lesion Mask")
    ax[3].imshow(border_img); ax[3].set_title(f"Diagnosis: {pred}\nSeverity: {sev:.2f}%")
    plt.show()

visualize_final(full_ds.samples[0][0], ensemble, full_ds.classes)